<a href="https://colab.research.google.com/github/lakshya701/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lakshya701/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*
**Unit of analysis:** One row = one content page (content_hash_id), for one client
(client_hash_id), on one report date — from fact_content_daily_performance. When I
aggregate for features, one row becomes one content page evaluated over a trailing
window.

**Time window:** I develop and test on a mid-panel month (month=2026-03) as my working
window — confirmed via query to run March 1 to March 31, 2026, with no overlap into
other months. I use a prior 90-day feature window to predict an outcome over the next
30 days, and I treat the final month in the release (June 2026) as a sealed test month
I do not touch while developing my label or features.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/lakshya701/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)

from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH_TABLE = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

grain_check = con.sql(f"SELECT client_hash_id, content_hash_id, report_date, gsc_impressions FROM {MONTH_TABLE} LIMIT 5").df()
print("Confirms grain: one row = one client + one content item + one report date")
grain_check

Confirms grain: one row = one client + one content item + one report date


,client_hash_id,content_hash_id,report_date,gsc_impressions
0,client_73cda7b4e4f265ea,content_b7e512995f79d5a6,2026-03-01,20
1,client_73cda7b4e4f265ea,content_05597932fe4da067,2026-03-01,1
2,client_73cda7b4e4f265ea,content_7a105f548d9c6916,2026-03-01,125
3,client_73cda7b4e4f265ea,content_905aa32a0230694e,2026-03-01,7
4,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,2026-03-01,11


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*
**Feature fields (knowable at prediction time):**
- gsc_impressions, gsc_clicks, gsc_avg_position — trailing observed search signals
- days_since_last_update — static fact about the content as of the decision date
- visible_query_count (from fact_content_query_90d) — trailing 90-day query-mix signal

**Label field (what I predict):**
- A future-window decline label: whether impressions drop by more than 20% over the
  next 30 days, compared to the prior 90-day baseline. Not shipped in the data — I
  calculate it myself from the outcome window, strictly after the feature window.

**Context fields (background, not features or labels):**
- content_type, main_intent, word_count — useful for grouping/interpreting results,
  not core predictive signals for this first pass.

**Excluded fields, with why:**
- Any reconstructed product decision flag (health_score, priority_score, action_type) —
  excluded because these aren't shipped in the data, and rebuilding one myself and
  feeding it back as a feature would just teach the model to copy an existing rule
  instead of finding real signal.
- trend_pct when used alongside a label built from trend_direction — excluded because
  it's literally the value the label is derived from; including it would be leakage.
- GA4-dependent engagement features (sessions, scroll rate) for this pass — excluded
  because only about 4.2% of rows in this month (413,966 of 9,841,378) have GA4 data
  available, so relying on it now would silently drop most of the dataset.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
import os, sys, subprocess, getpass

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/lakshya701/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb", "huggingface_hub"], check=True)

from google.colab import userdata
HF_TOKEN = userdata.get('HF_TOKEN')
print("Token loaded:", "YES" if HF_TOKEN else "EMPTY")

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH_TABLE = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"

# Query 1: grain + row count and date span
span = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS min_date, MAX(report_date) AS max_date
    FROM {MONTH_TABLE}
""").df()
print("\nRow count and date span:")
print(span)

# Query 2: missing values / availability check
missing = con.sql(f"""
    SELECT
        SUM(CASE WHEN gsc_impressions IS NULL THEN 1 ELSE 0 END) AS missing_impressions,
        SUM(CASE WHEN gsc_avg_position IS NULL THEN 1 ELSE 0 END) AS missing_position,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        COUNT(*) AS total_rows
    FROM {MONTH_TABLE}
""").df()
print("\nMissing values / availability check:")
print(missing)

# Query 3: confirm no window overlap - only March 2026 present
window_check = con.sql(f"""
    SELECT DISTINCT strftime(report_date, '%Y-%m') AS month
    FROM {MONTH_TABLE}
""").df()
print("\nConfirming this table only contains March 2026:")
print(window_check)

Token loaded: YES


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Row count and date span:
   row_count   min_date   max_date
0    9841378 2026-03-01 2026-03-31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Missing values / availability check:
   missing_impressions  missing_position  ga4_available_rows  total_rows
0                  0.0         6230317.0            413966.0     9841378


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


Confirming this table only contains March 2026:
     month
0  2026-03


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*
This data can never tell me whether a refresh *caused* a recovery (needs a real
experiment, not observational data), or anything about Google's actual ranking
algorithm. It also has a real gap I found in my own verification: only 413,966 out of
9,841,378 rows (about 4.2%) have GA4 data available in this month — meaning any feature
that relies on engagement/session data would only apply to a small slice of pages. The
panel is also unbalanced across clients; before trusting any engagement-based feature,
I'd need to check dim_clients.gsc_data_start and ga4_data_start per client rather than
assuming full coverage.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.